# ICU Handoff Summary Evaluation

Generated summaries (`output/results/P{N} summary.txt`) vs Gold Standard (`3단계_ Gold Standard Summary.xlsx`)

**Metrics**: BERTScore F1, ROUGE-L F1, SBERT Cosine Similarity

In [ ]:
# === Clone repo (Colab only) ===
import os
if not os.path.exists("emr_icu_handover"):
    !git clone https://github.com/taejun-song/icu_nurse_handoff.git emr_icu_handover
os.chdir("emr_icu_handover")

In [ ]:
# === Configuration ===
GOLD_XLSX_PATH = "3단계_ Gold Standard Summary.xlsx"  # relative to repo root
PRED_DIR = "output/results"                           # directory with P{N} summary.txt
OUTPUT_DIR = "output/eval_results"                    # where to save evaluation outputs

USE_SBERT = True  # set False to skip SBERT (saves time/memory)

BERTSCORE_MODEL = "bert-base-multilingual-cased"
SBERT_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "src").exists(), f"src/ not found in {REPO_ROOT}. Run the clone cell first."

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import display, Markdown
from src.alignment import load_gold_standard, load_predictions, parse_txt_sections, build_pairs
from src.eval import (
    compute_bertscore, compute_rouge_l, compute_sbert_cosine,
    aggregate_by_sheet, aggregate_overall, save_results,
)

GOLD_XLSX = REPO_ROOT / GOLD_XLSX_PATH
PRED = REPO_ROOT / PRED_DIR
OUT = REPO_ROOT / OUTPUT_DIR
print(f"Repo root: {REPO_ROOT}")
print(f"Gold XLSX: {GOLD_XLSX} (exists={GOLD_XLSX.exists()})")
print(f"Pred dir:  {PRED} (exists={PRED.exists()})")

In [ ]:
import sys
from pathlib import Path

_nb_dir = Path(globals().get("_dh", ["."])[0]).resolve()
REPO_ROOT = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
assert (REPO_ROOT / "src").exists(), f"src/ not found in {REPO_ROOT}. Run from project root."

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import display, Markdown
from src.alignment import load_gold_standard, load_predictions, parse_txt_sections, build_pairs
from src.eval import (
    compute_bertscore, compute_rouge_l, compute_sbert_cosine,
    aggregate_by_sheet, aggregate_overall, save_results,
)

GOLD_XLSX = REPO_ROOT / GOLD_XLSX_PATH
PRED = REPO_ROOT / PRED_DIR
OUT = REPO_ROOT / OUTPUT_DIR
print(f"Repo root: {REPO_ROOT}")
print(f"Gold XLSX: {GOLD_XLSX} (exists={GOLD_XLSX.exists()})")
print(f"Pred dir:  {PRED} (exists={PRED.exists()})")

In [ ]:
# === Inspect gold standard ===
gold_sheets = load_gold_standard(GOLD_XLSX)
print(f"Patient sheets: {list(gold_sheets.keys())} ({len(gold_sheets)} total)\n")
for name, df in gold_sheets.items():
    print(f"  {name}: {len(df)} rows, columns={list(df.columns[:3])}")

In [ ]:
# === Inspect predictions ===
predictions = load_predictions(PRED)
print(f"Prediction files found: {len(predictions)}\n")
for pid, text in sorted(predictions.items(), key=lambda x: int(x[0][1:])):
    print(f"  {pid}: {len(text)} chars")

In [ ]:
# === Build alignment pairs ===
pairs_df, unmatched_gold, unmatched_pred = build_pairs(GOLD_XLSX, PRED)
print(f"Total aligned pairs: {len(pairs_df)}")
print(f"Unmatched gold sheets: {unmatched_gold}")
print(f"Unmatched pred files:  {unmatched_pred}")
print(f"\nAlignment methods: {pairs_df['alignment_method'].value_counts().to_dict()}")

In [ ]:
# === Preview aligned pairs ===
preview = pairs_df[["sheet_name", "level_2", "gold_text_norm", "pred_text_norm"]].copy()
preview["gold_text_norm"] = preview["gold_text_norm"].str[:200]
preview["pred_text_norm"] = preview["pred_text_norm"].str[:200]
with pd.option_context("display.max_colwidth", 200, "display.max_rows", 30):
    display(preview.head(30))

In [ ]:
# === Compute metrics ===
golds = pairs_df["gold_text_norm"].tolist()
preds = pairs_df["pred_text_norm"].tolist()

print("Computing BERTScore...")
pairs_df["bertscore_f1"] = compute_bertscore(golds, preds, model_type=BERTSCORE_MODEL)
print(f"  done. mean={pairs_df['bertscore_f1'].mean():.4f}")

print("Computing ROUGE-L...")
pairs_df["rouge_l_f1"] = compute_rouge_l(golds, preds)
print(f"  done. mean={pairs_df['rouge_l_f1'].mean():.4f}")

if USE_SBERT:
    print("Computing SBERT cosine similarity...")
    pairs_df["sbert_cosine"] = compute_sbert_cosine(golds, preds, model_name=SBERT_MODEL)
    print(f"  done. mean={pairs_df['sbert_cosine'].mean():.4f}")

print("\nAll metrics computed.")

In [ ]:
# === Per-sample results ===
metric_cols = [c for c in pairs_df.columns if c.startswith(("bertscore", "rouge", "sbert"))]
display_cols = ["sheet_name", "level_1", "level_2"] + metric_cols
with pd.option_context("display.max_rows", 220, "display.float_format", "{:.4f}".format):
    display(pairs_df[display_cols])

In [ ]:
# === Per-sheet aggregation ===
sheet_agg = aggregate_by_sheet(pairs_df, metric_cols)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(sheet_agg)

In [ ]:
# === Overall aggregation ===
overall = aggregate_overall(pairs_df, metric_cols)
for col in metric_cols:
    s = overall[col]
    print(f"{col}: mean={s['mean']:.4f}  median={s['median']:.4f}  std={s['std']:.4f}")
print(f"\nTotal pairs: {overall['n_pairs']}")

In [ ]:
# === Save outputs ===
config = {
    "gold_xlsx": str(GOLD_XLSX),
    "pred_dir": str(PRED),
    "bertscore_model": BERTSCORE_MODEL,
    "sbert_model": SBERT_MODEL if USE_SBERT else None,
    "use_sbert": USE_SBERT,
}
save_results(pairs_df, sheet_agg, overall, OUT, config=config)
print(f"Results saved to {OUT}/")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")